In [1]:
%load_ext autoreload
%autoreload 2

In [15]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.core.problem import Problem
from pymoo.operators.crossover.ox import OrderCrossover
from pymoo.operators.mutation.inversion import InversionMutation
from pymoo.operators.sampling.rnd import PermutationRandomSampling
from pymoo.optimize import minimize
from pymoo.termination import get_termination
from pymoo.termination.collection import TerminationCollection
from pymoo.termination.default import DefaultSingleObjectiveTermination
from tqdm import tqdm

from qap_assignment.dataset import make_dataset, parse_dat_file

make_dataset()

* pymoo `Problem` needs some metadata https://pymoo.org/problems/definition.html
  * In QAP, our solutions are premutations of where facilities should go
  * $n$ facilities, so permutations ($p$ (QAPLIB) or $\varphi$ (PDF)) are size $n$ (`n_var = n`)
    * Use $p$ here b/c convenient, we'll use $\varphi$ in report
  * `n_obj = 1`, single objective b/c we're minimizing the objective function
    * QAPLIB problems don't have $C$, PDF we got says: "If there is no linear term (hence, no matrix $C$), we just write $QAP(A,\ B)$" (p. 204)
  * `n_constr` defaults to 0, omit b/c QAP doesn't have constraints
  * `xl` and `xu` are bounds of array indexes for arranging the locations
  * `vtype` is optional, but whatever we'll set it
    * FWIW, the TSP problem sets it https://github.com/anyoptimization/pymoo/blob/2aa012be7e6049f436453eaba95ae054fe6b5b65/pymoo/problems/single/traveling_salesman.py#L30
* We want `Problem` so its vectorized, so evaluation function handles whole population at once

In [16]:
class QAP(Problem):
    def __init__(self, n: int, A: np.ndarray, B: np.ndarray):
        super().__init__(n_var=n, n_obj=1, xl=0, xu=n - 1, vtype=int)
        self.A = A
        self.B = B

    def _evaluate(self, x, out, *args, **kwargs):
        # Docs say setting `count` is better
        costs = np.fromiter(
            (np.sum(self.A * self.B[p][:, p]) for p in x),
            dtype=np.int64,
            count=len(x),
        )
        # out['F'] neeeds to be (len(x), self.n_obj)
        out["F"] = costs.reshape(-1, 1)

Would like to try PMX and swap mutation, but we'd have to implement that,
and I'm lazy so just use OX and inversion.

In [26]:
def run(seed):
    n, A, B = parse_dat_file("kra30a")
    problem = QAP(n, A, B)

    algorithm = GA(
        pop_size=300,
        sampling=PermutationRandomSampling(),
        # `Crossover` uses prob=0.9 by default anyway
        crossover=OrderCrossover(prob=0.9),
        mutation=InversionMutation(prob=0.2),
        # By default, tournament selection and elitist survival is used
    )

    termination = TerminationCollection(
        # Default thing has other stuff so this will make it terminate early,
        # just here for testing stuff actually works, don't use
        # DefaultSingleObjectiveTermination(n_max_gen=50000),
        get_termination("n_gen", 50000),
        # Why is fmin not documented...
        # Shouldn't hardcode this, kra30a has a known optimum
        get_termination("fmin", 88900),
    )

    return minimize(problem, algorithm, termination, seed=seed, verbose=False)

In [27]:
tasks = (delayed(run)(i) for i in range(50))
results = []
for res in tqdm(
    Parallel(n_jobs=8, return_as="generator_unordered")(tasks),
    total=50,
):
    results.append(res)

100%|██████████| 50/50 [9:36:04<00:00, 691.28s/it]    


In [28]:
fs = [res.F for r in results]

In [29]:
fs

[array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.]),
 array([93350.])]